In [ ]:
# ============================================================
# Notebook 09 — Final LightGBM Improvement
# Cell 1 — Environment setup
# ============================================================

from pathlib import Path
import warnings
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import (
    LGBMRegressor,
    early_stopping,
    log_evaluation
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

DATA_FOLDER = Path("../data/favorita")

CLEAN_DATA_PATH = (
    DATA_FOLDER / "benchmark_300_cleaned.csv"
)

# ------------------------------------------------------------
# Frozen project settings
# ------------------------------------------------------------

GROUP_COLUMNS = ["store_nbr", "item_nbr"]

FORECAST_HORIZONS = [7, 14, 30, 60]
MAX_FORECAST_HORIZON = 60

print("Notebook 09 environment ready.")
print("Clean dataset:", CLEAN_DATA_PATH)
print("Forecast horizons:", FORECAST_HORIZONS)

In [ ]:
# ============================================================
# Cell 2 — Load cleaned multi-series benchmark data
# ============================================================

data = pd.read_csv(
    CLEAN_DATA_PATH,
    parse_dates=["date"]
)

# Keep the data in chronological order within each store-item series.
data = (
    data
    .sort_values(["store_nbr", "item_nbr", "date"])
    .reset_index(drop=True)
)

print("Shape:", data.shape)
print("Number of series:", data[GROUP_COLUMNS].drop_duplicates().shape[0])
print("Date range:", data["date"].min(), "to", data["date"].max())
print("Missing demand targets:", data["demand_target"].isna().sum())

data.head()

In [ ]:
# ============================================================
# Cell 3 — Add item and store metadata
# ============================================================

items = pd.read_csv(DATA_FOLDER / "items.csv")
stores = pd.read_csv(DATA_FOLDER / "stores.csv")

# Keep only the metadata we actually need.
items = items[
    ["item_nbr", "family", "class", "perishable"]
].copy()

stores = stores[
    ["store_nbr", "city", "state", "type", "cluster"]
].copy()

# Give "type" a clearer feature name.
stores = stores.rename(columns={"type": "store_type"})

# Merge metadata onto every store-item-date row.
model_data = (
    data
    .merge(items, on="item_nbr", how="left")
    .merge(stores, on="store_nbr", how="left")
)

print("Shape:", model_data.shape)
print("Series:", model_data[GROUP_COLUMNS].drop_duplicates().shape[0])

print("\nMissing metadata:")
print(
    model_data[
        ["family", "class", "perishable",
         "city", "state", "store_type", "cluster"]
    ].isna().sum()
)

In [ ]:
# ============================================================
# Cell 4 — Improved historical demand features
# ============================================================

model_data = model_data.sort_values(
    ["store_nbr", "item_nbr", "date"]
).reset_index(drop=True)

grouped = model_data.groupby(
    ["store_nbr", "item_nbr"],
    observed=True
)["demand_target"]

# Historical lags
for lag in [1, 7, 14, 21, 28, 35, 42, 56]:
    model_data[f"lag_{lag}"] = grouped.shift(lag)

# Use ONLY demand available before the current day.
previous_demand = grouped.shift(1)

group_keys = [
    model_data["store_nbr"],
    model_data["item_nbr"]
]

# Rolling demand behaviour
for window in [7, 14, 28, 56]:

    rolling = previous_demand.groupby(
        group_keys,
        observed=True
    ).rolling(window, min_periods=1)

    model_data[f"rolling_mean_{window}"] = (
        rolling.mean()
        .reset_index(level=[0, 1], drop=True)
    )

# Extra features aimed at detecting spikes / changing demand.
for window in [7, 28]:

    rolling = previous_demand.groupby(
        group_keys,
        observed=True
    ).rolling(window, min_periods=1)

    model_data[f"rolling_std_{window}"] = (
        rolling.std()
        .reset_index(level=[0, 1], drop=True)
    )

    model_data[f"rolling_median_{window}"] = (
        rolling.median()
        .reset_index(level=[0, 1], drop=True)
    )

    model_data[f"rolling_max_{window}"] = (
        rolling.max()
        .reset_index(level=[0, 1], drop=True)
    )

# Recent trend:
# > 1 means recent demand is stronger than the longer-term baseline.
model_data["demand_trend_7_28"] = (
    model_data["rolling_mean_7"] /
    (model_data["rolling_mean_28"] + 1.0)
)

print("Historical features created.")
print("Current shape:", model_data.shape)

In [ ]:
# ============================================================
# Cell 5 — Calendar and future-known business features
# ============================================================

# Basic calendar features
model_data["day_of_week"] = model_data["date"].dt.dayofweek
model_data["day_of_month"] = model_data["date"].dt.day
model_data["month"] = model_data["date"].dt.month
model_data["week_of_year"] = model_data["date"].dt.isocalendar().week.astype(int)

# Weekend indicator
model_data["is_weekend"] = (
    model_data["day_of_week"] >= 5
).astype(int)

# Ecuador-style payday signals:
# Demand may change around the middle and end of the month.
model_data["is_mid_month"] = (
    model_data["day_of_month"].between(14, 16)
).astype(int)

model_data["is_month_end"] = (
    model_data["date"].dt.is_month_end |
    model_data["day_of_month"].between(28, 31)
).astype(int)

# Convert existing boolean calendar columns to integers.
calendar_flags = [
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
]

for column in calendar_flags:
    model_data[column] = (
        model_data[column]
        .fillna(False)
        .astype(int)
    )

print("Calendar/business features created.")
print("Current shape:", model_data.shape)

print(
    model_data[
        [
            "date",
            "day_of_week",
            "day_of_month",
            "month",
            "week_of_year",
            "is_weekend",
            "is_mid_month",
            "is_month_end",
            "promotion_status",
        ]
    ].head()
)

In [ ]:
# ============================================================
# Cell 6 — Frozen temporal backtests
# ============================================================

BACKTESTS = [
    {
        "name": "Backtest 1",
        "train_end": pd.Timestamp("2017-01-01"),
        "forecast_start": pd.Timestamp("2017-01-02"),
    },
    {
        "name": "Backtest 2",
        "train_end": pd.Timestamp("2017-02-28"),
        "forecast_start": pd.Timestamp("2017-03-01"),
    },
    {
        "name": "Backtest 3",
        "train_end": pd.Timestamp("2017-04-30"),
        "forecast_start": pd.Timestamp("2017-05-01"),
    },
]

for bt in BACKTESTS:
    bt["forecast_end"] = (
        bt["forecast_start"]
        + pd.Timedelta(days=MAX_FORECAST_HORIZON - 1)
    )

    print(
        bt["name"],
        "| Train through:", bt["train_end"].date(),
        "| Forecast:",
        bt["forecast_start"].date(),
        "to",
        bt["forecast_end"].date()
    )

In [ ]:
# ============================================================
# Cell 7 — Direct multi-horizon training setup
# ============================================================

# These horizons cover short-, medium-, and long-range behaviour.
# We test the architecture cheaply first before expanding it.
DIRECT_TRAIN_HORIZONS = [1, 3, 7, 14, 21, 30, 45, 60]

direct_parts = []

for horizon in DIRECT_TRAIN_HORIZONS:

    temp = model_data.copy()

    # Tell the model how far into the future it is predicting.
    temp["forecast_horizon"] = horizon

    # Future demand becomes the target.
    temp["future_demand"] = (
        temp.groupby(
            ["store_nbr", "item_nbr"],
            observed=True
        )["demand_target"]
        .shift(-horizon)
    )

    # The date whose demand we are predicting.
    temp["target_date"] = (
        temp["date"] + pd.Timedelta(days=horizon)
    )

    # Remove rows where the future target is unavailable.
    temp = temp[temp["future_demand"].notna()].copy()

    direct_parts.append(temp)

direct_data = pd.concat(
    direct_parts,
    ignore_index=True
)

del direct_parts

print("Direct training rows:", f"{len(direct_data):,}")
print("Horizons:", sorted(direct_data["forecast_horizon"].unique()))
print(
    "Target range:",
    direct_data["future_demand"].min(),
    "to",
    direct_data["future_demand"].max()
)

In [ ]:
# ============================================================
# Cell 8 — Future-known features for the target date
# ============================================================

future_columns = [
    "store_nbr",
    "item_nbr",
    "date",
    "promotion_status",
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
    "day_of_week",
    "day_of_month",
    "month",
    "week_of_year",
    "is_weekend",
    "is_mid_month",
    "is_month_end",
]

future_info = model_data[future_columns].copy()

# Rename these so it is explicit that they describe
# the day we are forecasting, not the forecast origin.
future_info = future_info.rename(
    columns={
        "date": "target_date",
        **{
            col: f"target_{col}"
            for col in future_columns
            if col not in ["store_nbr", "item_nbr", "date"]
        }
    }
)

direct_data = direct_data.merge(
    future_info,
    on=["store_nbr", "item_nbr", "target_date"],
    how="left"
)

target_feature_columns = [
    c for c in direct_data.columns
    if c.startswith("target_")
]

print("Direct data shape:", direct_data.shape)
print("Future-known features:", len(target_feature_columns))
print(
    "Missing target-date matches:",
    direct_data["target_day_of_week"].isna().sum()
)

In [ ]:
# ============================================================
# Cell 9 — Final feature lists for direct LightGBM
# ============================================================

HISTORICAL_FEATURES = [
    "lag_1", "lag_7", "lag_14", "lag_21",
    "lag_28", "lag_35", "lag_42", "lag_56",

    "rolling_mean_7", "rolling_mean_14",
    "rolling_mean_28", "rolling_mean_56",

    "rolling_std_7", "rolling_std_28",
    "rolling_median_7", "rolling_median_28",
    "rolling_max_7", "rolling_max_28",

    "demand_trend_7_28",
]

STATIC_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "perishable",
    "city",
    "state",
    "store_type",
    "cluster",
]

FUTURE_FEATURES = [
    "target_promotion_status",
    "target_is_holiday",
    "target_is_event",
    "target_is_additional",
    "target_is_transfer",
    "target_is_bridge",
    "target_is_work_day",
    "target_day_of_week",
    "target_day_of_month",
    "target_month",
    "target_week_of_year",
    "target_is_weekend",
    "target_is_mid_month",
    "target_is_month_end",
]

DIRECT_FEATURES = (
    STATIC_FEATURES
    + HISTORICAL_FEATURES
    + FUTURE_FEATURES
    + ["forecast_horizon"]
)

CATEGORICAL_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "state",
    "store_type",
    "cluster",
    "target_promotion_status",
    "target_day_of_week",
    "target_month",
]

# LightGBM handles categorical columns efficiently when
# pandas marks them explicitly as category dtype.
for column in CATEGORICAL_FEATURES:
    direct_data[column] = direct_data[column].astype("category")

print("Total model features:", len(DIRECT_FEATURES))
print("Historical:", len(HISTORICAL_FEATURES))
print("Static:", len(STATIC_FEATURES))
print("Future-known:", len(FUTURE_FEATURES))

In [ ]:
# ============================================================
# Cell 10 — Quick temporal validation of direct architecture
# ============================================================

# Use Backtest 3 because it is the most recent of our
# three development backtests.
VALIDATION_START = pd.Timestamp("2017-05-01")
VALIDATION_END = pd.Timestamp("2017-06-29")

# IMPORTANT:
# Training examples are allowed only when BOTH the forecast
# origin and the future target occur before validation begins.
train_mask = (
    (direct_data["date"] < VALIDATION_START)
    & (direct_data["target_date"] < VALIDATION_START)
)

valid_mask = (
    direct_data["target_date"].between(
        VALIDATION_START,
        VALIDATION_END
    )
    & (direct_data["date"] < VALIDATION_START)
)

train_df = direct_data.loc[train_mask].copy()
valid_df = direct_data.loc[valid_mask].copy()

print("Training rows:", f"{len(train_df):,}")
print("Validation rows:", f"{len(valid_df):,}")

model = LGBMRegressor(
    objective="regression_l1",
    learning_rate=0.03,
    n_estimators=1500,
    num_leaves=63,
    min_child_samples=50,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

start_time = time.time()

model.fit(
    train_df[DIRECT_FEATURES],
    train_df["future_demand"],
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[
        (
            valid_df[DIRECT_FEATURES],
            valid_df["future_demand"]
        )
    ],
    eval_metric="mae",
    callbacks=[
        early_stopping(100, verbose=False),
        log_evaluation(0),
    ],
)

pred = np.clip(
    model.predict(valid_df[DIRECT_FEATURES]),
    0,
    None
)

actual = valid_df["future_demand"].to_numpy()

mae = np.mean(np.abs(actual - pred))
wape = (
    np.abs(actual - pred).sum()
    / np.abs(actual).sum()
    * 100
)

print("\nBest iteration:", model.best_iteration_)
print("Validation MAE:", round(mae, 4))
print("Validation WAPE:", round(wape, 4), "%")
print("Runtime:", round((time.time() - start_time) / 60, 2), "minutes")

In [ ]:
# ============================================================
# Cell 11 — Fair 60-day direct forecast: Backtest 3
# ============================================================

BT3_TRAIN_END = pd.Timestamp("2017-04-30")
BT3_FORECAST_START = pd.Timestamp("2017-05-01")

# One forecast-origin row per series.
origin = model_data[
    model_data["date"] == BT3_TRAIN_END
].copy()

print("Origin series:", len(origin))

# Expand each series into horizons 1...60.
horizons = pd.DataFrame({
    "forecast_horizon": np.arange(1, 61)
})

forecast_input = origin.merge(
    horizons,
    how="cross"
)

forecast_input["target_date"] = (
    forecast_input["date"]
    + pd.to_timedelta(
        forecast_input["forecast_horizon"],
        unit="D"
    )
)

# ------------------------------------------------------------
# Add future-known target-date features
# ------------------------------------------------------------

future_info_bt3 = model_data[future_columns].copy()

future_info_bt3 = future_info_bt3.rename(
    columns={
        "date": "target_date",
        **{
            col: f"target_{col}"
            for col in future_columns
            if col not in ["store_nbr", "item_nbr", "date"]
        }
    }
)

forecast_input = forecast_input.merge(
    future_info_bt3,
    on=["store_nbr", "item_nbr", "target_date"],
    how="left"
)

# Match categorical dtypes used during training.
for column in CATEGORICAL_FEATURES:
    forecast_input[column] = pd.Categorical(
        forecast_input[column],
        categories=direct_data[column].cat.categories
    )

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

forecast_input["prediction"] = np.clip(
    model.predict(
        forecast_input[DIRECT_FEATURES]
    ),
    0,
    None
)

# ------------------------------------------------------------
# Attach actual demand
# ------------------------------------------------------------

actuals = model_data[
    ["store_nbr", "item_nbr", "date", "demand_target"]
].rename(
    columns={
        "date": "target_date",
        "demand_target": "actual"
    }
)

forecast_eval = forecast_input.merge(
    actuals,
    on=["store_nbr", "item_nbr", "target_date"],
    how="left"
)

forecast_eval = forecast_eval[
    forecast_eval["actual"].notna()
].copy()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

direct_mae = np.mean(
    np.abs(
        forecast_eval["actual"]
        - forecast_eval["prediction"]
    )
)

direct_wape = (
    np.abs(
        forecast_eval["actual"]
        - forecast_eval["prediction"]
    ).sum()
    / np.abs(forecast_eval["actual"]).sum()
    * 100
)

print("\nRows evaluated:", len(forecast_eval))
print("Direct 60-day MAE:", round(direct_mae, 4))
print("Direct 60-day WAPE:", round(direct_wape, 4), "%")

print("\nOld recursive BT3:")
print("MAE:  3.1676")
print("WAPE: 43.4885 %")

In [12]:
# ============================================================
# Cell 12 — Objective test: attack prediction smoothing
# ============================================================

RECURSIVE_FEATURES = (
    STATIC_FEATURES
    + HISTORICAL_FEATURES
    + [
        "promotion_status",
        "is_holiday",
        "is_event",
        "is_additional",
        "is_transfer",
        "is_bridge",
        "is_work_day",
        "day_of_week",
        "day_of_month",
        "month",
        "week_of_year",
        "is_weekend",
        "is_mid_month",
        "is_month_end",
    ]
)

RECURSIVE_CATEGORICALS = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "state",
    "store_type",
    "cluster",
    "promotion_status",
    "day_of_week",
    "month",
]

# Make categorical columns explicit.
for column in RECURSIVE_CATEGORICALS:
    model_data[column] = model_data[column].astype("category")

# One-step temporal validation isolates objective behaviour.
train = model_data[
    (model_data["date"] < VALIDATION_START)
    & model_data["demand_target"].notna()
].copy()

valid = model_data[
    model_data["date"].between(
        VALIDATION_START,
        VALIDATION_END
    )
    & model_data["demand_target"].notna()
].copy()

OBJECTIVES = {
    "L1": {
        "objective": "regression_l1"
    },
    "L2": {
        "objective": "regression_l2"
    },
    "Tweedie": {
        "objective": "tweedie",
        "tweedie_variance_power": 1.3
    }
}

objective_results = []

for name, extra_params in OBJECTIVES.items():

    test_model = LGBMRegressor(
        learning_rate=0.03,
        n_estimators=1200,
        num_leaves=63,
        min_child_samples=50,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **extra_params
    )

    start = time.time()

    test_model.fit(
        train[RECURSIVE_FEATURES],
        train["demand_target"],
        categorical_feature=RECURSIVE_CATEGORICALS,
        eval_set=[
            (
                valid[RECURSIVE_FEATURES],
                valid["demand_target"]
            )
        ],
        eval_metric="mae",
        callbacks=[
            early_stopping(100, verbose=False),
            log_evaluation(0),
        ],
    )

    predictions = np.clip(
        test_model.predict(valid[RECURSIVE_FEATURES]),
        0,
        None
    )

    actual = valid["demand_target"].to_numpy()

    mae = np.mean(np.abs(actual - predictions))

    wape = (
        np.abs(actual - predictions).sum()
        / np.abs(actual).sum()
        * 100
    )

    objective_results.append({
        "objective": name,
        "MAE": mae,
        "WAPE": wape,
        "best_iteration": test_model.best_iteration_,
        "runtime_min": (time.time() - start) / 60
    })

objective_results = pd.DataFrame(
    objective_results
).sort_values("WAPE")

objective_results

,objective,MAE,WAPE,best_iteration,runtime_min
0,L1,3.013789,41.376532,1169,0.45624
2,Tweedie,3.114229,42.755490,1017,0.33314
1,L2,3.222324,44.239530,367,0.13020


In [13]:
# ============================================================
# Cell 13 — Add stronger predictive features
# ============================================================

# Recent demand acceleration
model_data["trend_7_14"] = (
    model_data["rolling_mean_7"]
    / (model_data["rolling_mean_14"] + 1.0)
)

model_data["trend_14_28"] = (
    model_data["rolling_mean_14"]
    / (model_data["rolling_mean_28"] + 1.0)
)

# Distance from recent maximum
model_data["max_ratio_7"] = (
    model_data["rolling_max_7"]
    / (model_data["rolling_mean_7"] + 1.0)
)

model_data["max_ratio_28"] = (
    model_data["rolling_max_28"]
    / (model_data["rolling_mean_28"] + 1.0)
)

# Same-weekday demand behaviour
weekday_group = model_data.groupby(
    ["store_nbr", "item_nbr", "day_of_week"],
    observed=True
)["demand_target"]

model_data["same_weekday_lag_1"] = weekday_group.shift(1)

model_data["same_weekday_mean_4"] = (
    model_data["same_weekday_lag_1"]
    .groupby([
        model_data["store_nbr"],
        model_data["item_nbr"],
        model_data["day_of_week"],
    ])
    .rolling(4, min_periods=1)
    .mean()
    .reset_index(level=[0, 1, 2], drop=True)
)

# Recent zero-demand behaviour
previous_demand = model_data.groupby(
    ["store_nbr", "item_nbr"],
    observed=True
)["demand_target"].shift(1)

model_data["zero_rate_28"] = (
    previous_demand.eq(0)
    .groupby([
        model_data["store_nbr"],
        model_data["item_nbr"],
    ])
    .rolling(28, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

print("New predictive features added.")
print("Shape:", model_data.shape)

New predictive features added.
Shape: (480213, 58)


In [15]:
# ============================================================
# Cell 14 — Feature improvement validation
# ============================================================

NEW_FEATURES = [
    "trend_7_14",
    "trend_14_28",
    "max_ratio_7",
    "max_ratio_28",
    "same_weekday_lag_1",
    "same_weekday_mean_4",
    "zero_rate_28",
]

BASE_FEATURES = RECURSIVE_FEATURES.copy()
IMPROVED_FEATURES = BASE_FEATURES + NEW_FEATURES

# Recreate train/validation AFTER adding the new features.
train = model_data[
    (model_data["date"] < VALIDATION_START)
    & model_data["demand_target"].notna()
].copy()

valid = model_data[
    model_data["date"].between(
        VALIDATION_START,
        VALIDATION_END
    )
    & model_data["demand_target"].notna()
].copy()

feature_test_results = []

for name, features in {
    "Base": BASE_FEATURES,
    "Improved": IMPROVED_FEATURES,
}.items():

    test_model = LGBMRegressor(
        objective="regression_l1",
        learning_rate=0.03,
        n_estimators=1400,
        num_leaves=63,
        min_child_samples=50,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )

    start = time.time()

    test_model.fit(
        train[features],
        train["demand_target"],
        categorical_feature=RECURSIVE_CATEGORICALS,
        eval_set=[
            (
                valid[features],
                valid["demand_target"]
            )
        ],
        eval_metric="mae",
        callbacks=[
            early_stopping(100, verbose=False),
            log_evaluation(0),
        ],
    )

    pred = np.clip(
        test_model.predict(valid[features]),
        0,
        None
    )

    actual = valid["demand_target"].to_numpy()

    mae = np.mean(np.abs(actual - pred))

    wape = (
        np.abs(actual - pred).sum()
        / np.abs(actual).sum()
        * 100
    )

    feature_test_results.append({
        "feature_set": name,
        "MAE": mae,
        "WAPE": wape,
        "best_iteration": test_model.best_iteration_,
        "runtime_min": (time.time() - start) / 60,
    })

feature_test_results = pd.DataFrame(
    feature_test_results
).sort_values("WAPE")

feature_test_results

,feature_set,MAE,WAPE,best_iteration,runtime_min
0,Base,3.012043,41.352563,1394,0.484663
1,Improved,3.018398,41.439816,1400,0.500981
